# ASRS LoRA Classifier - Preprocessing

### First look at the data - loading one file and looking at it

In [1]:
import pandas as pd

DATA_FOLDER = "data/raw/nasa/"
path = "data/raw/nasa/asrs_2010_H1.csv"
raw = pd.read_csv(path, header=None, dtype=str, nrows=5)
print(raw.shape)
raw.iloc[:, :6]

(5, 126)


,0,1,2,3,4,5
0,,Time,Time,Place,Place,Place
1,ACN,Date,Local Time Of Day,Locale Reference,State Reference,Relative Position.Angle.Radial
2,866849,201001,1201-1800,ZHU.ARTCC,TX,NaN
3,866859,201001,1201-1800,ZAU.ARTCC,IL,NaN
4,866866,201001,1801-2400,ZHU.ARTCC,TX,NaN


### We build the column names from the groups (row 0) and the fields (row 1)
By taking row 0 and 1, pairing them column by column and transforming each pair to a column name of the form {group} | {field}

Example:
- Row 0, Col 2 = Time
- Row 1, Col 2 = Local Time Of Day
- Therefore we get the new column name: "Time | Local Time Of Day"


In [2]:
print(repr(raw.iloc[0, 0]))

groups = raw.iloc[0].str.strip()
fields = raw.iloc[1].str.strip()

names = []
for g, f in zip(groups, fields):
    if g == "":
        names.append(f)
    else:
        names.append(f"{g} | {f}")

names[:6]

' '


['ACN',
 'Time | Date',
 'Time | Local Time Of Day',
 'Place | Locale Reference',
 'Place | State Reference',
 'Place | Relative Position.Angle.Radial']

In [3]:
len(names) == len(set(names))

True

In [4]:
[n for n in names if "Anomaly" in n or "Narrative" in n]

['Events | Anomaly', 'Report 1 | Narrative', 'Report 2 | Narrative']

### Load the full file, real column names, only what we need

In [5]:
df = pd.read_csv(path, header=None, skiprows=2, names=names, dtype=str)
df.shape

(2814, 126)

In [6]:
cols = {
    "ACN": "acn",
    "Time | Date": "date",
    "Events | Anomaly": "anomaly",
    "Report 1 | Narrative": "narrative",
}
df = df[list(cols)].rename(columns=cols)
df.head()

,acn,date,anomaly,narrative
0,866849,201001,ATC Issue All Types; Deviation / Discrepancy -...,Ocean West and Offshore West/Central were comb...
1,866859,201001,ATC Issue All Types; Deviation / Discrepancy -...,Aircraft X had a flight plan in URET (user req...
2,866866,201001,ATC Issue All Types; Airspace Violation All Ty...,I was working D-79 when Aircraft X reported ov...
3,866977,201001,Aircraft Equipment Problem Critical,Descending to FL340 we got a Green Hydraulic L...
4,866984,201001,Aircraft Equipment Problem Critical; Inflight ...,On climbout at approximately 15;000 FT noticed...


In [7]:
df.isna().sum()

acn           0
date          0
anomaly      12
narrative     0
dtype: int64

We can see that we get 12 missing anomalies. These rows will be dropped in the next step.

### From the anomaly string to a list of labels

In [8]:
# Drop the 12 reports with no anomaly
df = df.dropna(subset=["anomaly"])
df.shape

(2802, 4)

In [9]:
len(set(df["anomaly"]))

685

In [10]:
ORPHANS = {"Critical", "Less Severe"}

def parse_anomaly(s):
    labels = []
    for part in s.split(";"):
        part = part.strip()
        if part == "":
            continue
        if part in ORPHANS:
            assert labels[-1] == "Conflict Ground Conflict", (s, part)
            labels[-1] = f"{labels[-1]} {part}"
        else:
            labels.append(part)
    return labels

In [11]:
parse_anomaly("ATC Issue All Types; Conflict Ground Conflict; Critical; Ground Incursion Runway")

['ATC Issue All Types',
 'Conflict Ground Conflict Critical',
 'Ground Incursion Runway']

In [12]:
df["labels"] = df["anomaly"].apply(parse_anomaly)
df["labels"].explode().value_counts()

labels
Deviation / Discrepancy - Procedural Published Material / Policy     1014
Aircraft Equipment Problem Critical                                   868
ATC Issue All Types                                                   680
Aircraft Equipment Problem Less Severe                                435
Deviation / Discrepancy - Procedural Clearance                        386
Deviation - Track / Heading All Types                                 213
Conflict Airborne Conflict                                            212
Inflight Event / Encounter Weather / Turbulence                       156
Deviation / Discrepancy - Procedural FAR                              140
Deviation / Discrepancy - Procedural Maintenance                      112
Deviation - Altitude Excursion From Assigned Altitude                 107
Airspace Violation All Types                                           93
Conflict NMAC                                                          93
Inflight Event / Encounter Loss

In [13]:
counts = df["labels"].explode().value_counts()
"Critical" in counts.index, "Less Severe" in counts.index

(False, False)

### One function for one file

In [14]:
from pathlib import Path

def load_file(path):
    header = pd.read_csv(path, header=None, dtype=str, nrows=2)
    groups = header.iloc[0].str.strip()
    fields = header.iloc[1].str.strip()
    names = [f if g == "" else f"{g} | {f}" for g, f in zip(groups, fields)]

    df = pd.read_csv(path, header=None, skiprows=2, names=names, dtype=str)
    df = df[list(cols)].rename(columns=cols)
    df["file"] = path.name
    return df

In [15]:
load_file(Path(path)).shape

(2814, 5)

### All 32 files

In [16]:
paths = sorted(Path("data/raw/nasa").glob("*.csv"))
len(paths)

32

In [17]:
frames = [load_file(p) for p in paths]
all_df = pd.concat(frames, ignore_index=True)
all_df.shape

(85104, 5)

### Look at duplicates

In [18]:
all_df["acn"].duplicated().sum()

np.int64(0)

In [19]:
dups = all_df[all_df["acn"].duplicated(keep=False)].sort_values("acn")
dups.head(6)

,acn,date,anomaly,narrative,file


### Clean the full table and check the dates

In [20]:
all_df.isna().sum()

acn           0
date          0
anomaly      61
narrative     0
file          0
dtype: int64

In [21]:
all_df = all_df.dropna(subset=["anomaly", "narrative"])
all_df["labels"] = all_df["anomaly"].apply(parse_anomaly)
all_df.shape

(85043, 6)

In [22]:
all_df["year"] = all_df["date"].str[:4].astype(int)
file_year = all_df["file"].str[5:9].astype(int)
(file_year != all_df["year"]).sum()

np.int64(26)

In [23]:
all_df.groupby("year").size()

year
2002       1
2010    5483
2011    5635
2012    5053
2013    4477
2014    4574
2015    5972
2016    5406
2017    5211
2018    5657
2019    6426
2020    4981
2021    4571
2022    5451
2023    4590
2024    5427
2025    6128
dtype: int64

We found that there are 26 rows that show a missmatch between their year and the year in their filename. Therefore we will look at them.

In [24]:
mismatch = all_df[file_year != all_df["year"]]
mismatch[["acn", "date", "file"]]

,acn,date,file
31233,1777197,202012,asrs_2015_H2.csv
55888,1780556,202101,asrs_2020_H1.csv
55889,1780849,202101,asrs_2020_H1.csv
55890,1782173,202101,asrs_2020_H1.csv
55894,1790003,202102,asrs_2020_H1.csv
55897,1795476,202101,asrs_2020_H1.csv
55898,1813200,202106,asrs_2020_H1.csv
55899,1818480,202106,asrs_2020_H1.csv
58933,1801444,202108,asrs_2020_H2.csv
58935,1823047,202107,asrs_2020_H2.csv


In [25]:
all_df = all_df[all_df["year"] >= 2010]
all_df.shape

(85042, 7)

In [26]:
all_df

,acn,date,anomaly,narrative,file,labels,year
0,866849,201001,ATC Issue All Types; Deviation / Discrepancy -...,Ocean West and Offshore West/Central were comb...,asrs_2010_H1.csv,"[ATC Issue All Types, Deviation / Discrepancy ...",2010
1,866859,201001,ATC Issue All Types; Deviation / Discrepancy -...,Aircraft X had a flight plan in URET (user req...,asrs_2010_H1.csv,"[ATC Issue All Types, Deviation / Discrepancy ...",2010
2,866866,201001,ATC Issue All Types; Airspace Violation All Ty...,I was working D-79 when Aircraft X reported ov...,asrs_2010_H1.csv,"[ATC Issue All Types, Airspace Violation All T...",2010
3,866977,201001,Aircraft Equipment Problem Critical,Descending to FL340 we got a Green Hydraulic L...,asrs_2010_H1.csv,[Aircraft Equipment Problem Critical],2010
4,866984,201001,Aircraft Equipment Problem Critical; Inflight ...,On climbout at approximately 15;000 FT noticed...,asrs_2010_H1.csv,"[Aircraft Equipment Problem Critical, Inflight...",2010
...,...,...,...,...,...,...,...
85099,2349284,202507,Aircraft Equipment Problem Critical; Deviation...,On a flight from ZZZ to ZZZ1; I was flying nor...,asrs_2025_H2.csv,"[Aircraft Equipment Problem Critical, Deviatio...",2025
85100,2356648,202507,Aircraft Equipment Problem Critical,The incident occurred during a training flight...,asrs_2025_H2.csv,[Aircraft Equipment Problem Critical],2025
85101,2375013,202508,Deviation / Discrepancy - Procedural Published...,On arrival to ZZZZ ramp; at arrival; I misunde...,asrs_2025_H2.csv,[Deviation / Discrepancy - Procedural Publishe...,2025
85102,2384391,202509,Deviation / Discrepancy - Procedural FAR; Devi...,Flew a drone for a demo to introduce intern st...,asrs_2025_H2.csv,"[Deviation / Discrepancy - Procedural FAR, Dev...",2025
